# Build the Return-Risk Training Dataset
## Notebook purpose

This notebook combines:

- Main order data
- Leakage-safe customer history features
- Product and item features

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", "{:,.2f}".format)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
FEATURES_DIR = DATA_DIR / "features"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature directory: {FEATURES_DIR}")
print(f"Processed-data directory: {PROCESSED_DIR}")

Project root: C:\Users\Victus\OneDrive\Desktop\E-Commerce-Sales-Analytics
Feature directory: C:\Users\Victus\OneDrive\Desktop\E-Commerce-Sales-Analytics\data\features
Processed-data directory: C:\Users\Victus\OneDrive\Desktop\E-Commerce-Sales-Analytics\data\processed


# Load Main Orders And Engineered Feature Tables

In [2]:
orders_df = pd.read_csv(
    DATA_DIR / "ecommerce_sales_customer_analytics_150k.csv"
)

customer_features_df = pd.read_csv(
    FEATURES_DIR / "customer_features.csv"
)

product_features_df = pd.read_csv(
    FEATURES_DIR / "product_features.csv"
)

print(f"Main order rows:         {len(orders_df):,}")
print(f"Customer feature rows:   {len(customer_features_df):,}")
print(f"Product feature rows:    {len(product_features_df):,}")

Main order rows:         138,116
Customer feature rows:   138,116
Product feature rows:    138,116


# Validate Feature-Table Keys

In [3]:
order_ids = set(orders_df["order_id"])
customer_feature_order_ids = set(customer_features_df["order_id"])
product_feature_order_ids = set(product_features_df["order_id"])

merge_key_check = pd.Series({
    "main_orders": len(order_ids),
    "customer_feature_orders": len(customer_feature_order_ids),
    "product_feature_orders": len(product_feature_order_ids),
    "duplicate_customer_feature_order_ids": (
        customer_features_df["order_id"].duplicated().sum()
    ),
    "duplicate_product_feature_order_ids": (
        product_features_df["order_id"].duplicated().sum()
    ),
    "orders_missing_customer_features": len(
        order_ids - customer_feature_order_ids
    ),
    "orders_missing_product_features": len(
        order_ids - product_feature_order_ids
    ),
})

display(merge_key_check)

assert merge_key_check["duplicate_customer_feature_order_ids"] == 0
assert merge_key_check["duplicate_product_feature_order_ids"] == 0
assert merge_key_check["orders_missing_customer_features"] == 0
assert merge_key_check["orders_missing_product_features"] == 0

main_orders                             138116
customer_feature_orders                 138116
product_feature_orders                  138116
duplicate_customer_feature_order_ids         0
duplicate_product_feature_order_ids          0
orders_missing_customer_features             0
orders_missing_product_features              0
dtype: int64

# Create Timestamp And Return Label

In [4]:
orders_df["order_timestamp"] = pd.to_datetime(
    orders_df["order_date"].astype(str)
    + " "
    + orders_df["order_time"].astype(str),
    errors="coerce"
)

orders_df["return_label"] = (
    orders_df["return_status"] == "Returned"
).astype(int)

if orders_df["order_timestamp"].isna().sum() > 0:
    raise ValueError("Invalid order timestamps found.")

display(
    orders_df[
        [
            "order_id",
            "order_timestamp",
            "order_status",
            "return_status",
            "return_label",
        ]
    ].head()
)

,order_id,order_timestamp,order_status,return_status,return_label
0,ORD-301242,2023-11-06 16:37:47,Completed,NaN,0
1,ORD-773460,2025-12-24 01:22:36,Completed,NaN,0
2,ORD-449374,2021-07-05 14:24:21,Completed,NaN,0
3,ORD-567636,2023-01-21 07:20:26,Completed,NaN,0
4,ORD-820028,2022-05-13 09:46:21,Completed,NaN,0


# Select Eligible Orders


In [5]:
eligible_order_statuses = ["Completed", "Returned"]

eligible_orders_df = orders_df[
    orders_df["order_status"].isin(eligible_order_statuses)
].copy()

eligibility_summary = (
    orders_df
    .groupby("order_status", dropna=False)
    .agg(
        total_orders=("order_id", "count"),
        returned_orders=("return_label", "sum")
    )
    .reset_index()
)

eligibility_summary["included_in_training"] = (
    eligibility_summary["order_status"]
    .isin(eligible_order_statuses)
)

display(eligibility_summary)

print(f"Eligible training orders: {len(eligible_orders_df):,}")
print(
    "Eligible return rate: "
    f"{eligible_orders_df['return_label'].mean() * 100:.2f}%"
)

,order_status,total_orders,returned_orders,included_in_training
0,Cancelled,8398,0,False
1,Completed,113559,0,True
2,Pending,6697,0,False
3,Returned,9462,9462,True


Eligible training orders: 123,021
Eligible return rate: 7.69%


# Create Safe Order-Level Features


In [6]:
eligible_orders_df["order_month"] = (
    eligible_orders_df["order_timestamp"].dt.month
)

eligible_orders_df["order_day_of_week"] = (
    eligible_orders_df["order_timestamp"].dt.dayofweek
)

eligible_orders_df["order_hour"] = (
    eligible_orders_df["order_timestamp"].dt.hour
)

eligible_orders_df["order_quarter"] = (
    eligible_orders_df["order_timestamp"].dt.quarter
)

safe_order_feature_columns = [
    "order_id",
    "order_timestamp",
    "return_label",
    "order_month",
    "order_day_of_week",
    "order_hour",
    "order_quarter",
    "sales_channel",
    "payment_method",
    "currency",
    "shipping_method",
    "warehouse",
    "marketing_channel",
    "campaign_name",
    "coupon_code",
]

order_training_df = eligible_orders_df[
    safe_order_feature_columns
].copy()

display(order_training_df.head())

,order_id,order_timestamp,return_label,order_month,order_day_of_week,order_hour,order_quarter,sales_channel,payment_method,currency,shipping_method,warehouse,marketing_channel,campaign_name,coupon_code
0,ORD-301242,2023-11-06 16:37:47,0,11,0,16,4,Mobile App,Digital Wallet,USD,Standard,WH-003,Direct,Default_Campaign,NaN
1,ORD-773460,2025-12-24 01:22:36,0,12,2,1,4,Website,Debit Card,EUR,Economy,WH-005,Direct,Default_Campaign,NaN
2,ORD-449374,2021-07-05 14:24:21,0,7,0,14,3,Mobile App,Cash on Delivery,USD,Express,WH-015,YouTube,NaN,NaN
3,ORD-567636,2023-01-21 07:20:26,0,1,5,7,1,Social Media,Debit Card,USD,Express,WH-010,Email Marketing,NaN,NaN
4,ORD-820028,2022-05-13 09:46:21,0,5,4,9,2,Social Media,Digital Wallet,INR,Standard,WH-013,Direct,NaN,NaN


# Merge Customer And Product Features

In [7]:
training_df = order_training_df.merge(
    customer_features_df,
    on="order_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_customer"),
)

training_df = training_df.merge(
    product_features_df,
    on="order_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_product"),
)

print(f"Final training rows: {len(training_df):,}")
print(f"Final training columns: {training_df.shape[1]:,}")

Final training rows: 123,021
Final training columns: 55


# Verify Merge Quality

In [8]:
merge_validation = pd.Series({
    "training_rows": len(training_df),
    "unique_order_ids": training_df["order_id"].nunique(),
    "duplicate_order_ids": training_df["order_id"].duplicated().sum(),
    "missing_customer_age": training_df["customer_age"].isna().sum(),
    "missing_item_count": training_df["item_count"].isna().sum(),
    "returned_orders": training_df["return_label"].sum(),
    "non_returned_orders": (
        training_df["return_label"] == 0
    ).sum(),
    "return_rate_percent": round(
        training_df["return_label"].mean() * 100,
        2
    ),
})

display(merge_validation)

assert merge_validation["training_rows"] == merge_validation["unique_order_ids"]
assert merge_validation["duplicate_order_ids"] == 0
assert merge_validation["missing_customer_age"] == 0
assert merge_validation["missing_item_count"] == 0

training_rows          123,021.00
unique_order_ids       123,021.00
duplicate_order_ids          0.00
missing_customer_age         0.00
missing_item_count           0.00
returned_orders          9,462.00
non_returned_orders    113,559.00
return_rate_percent          7.69
dtype: float64

# Handle Missing Categorical Values

In [9]:
categorical_columns = training_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

categorical_columns = [
    column
    for column in categorical_columns
    if column not in ["order_id"]
]

training_df[categorical_columns] = (
    training_df[categorical_columns]
    .fillna("Unknown")
    .astype(str)
)

missing_summary = (
    training_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(training_df) * 100
).round(2)

display(missing_summary[missing_summary["missing_count"] > 0])

,missing_count,missing_percentage
prior_return_rate,22204,18.05
prior_average_order_value,22204,18.05
prior_average_discount,22204,18.05
days_since_previous_order,22204,18.05


# Final Feature Review


In [10]:
non_feature_columns = [
    "order_id",
    "order_timestamp",
    "return_label",
    "customer_id",
    "order_timestamp_customer",
]

feature_columns = [
    column
    for column in training_df.columns
    if column not in non_feature_columns
]

print(f"Number of model feature columns: {len(feature_columns)}")

print("\nFirst 20 feature columns:")
print(feature_columns[:20])

print("\nFinal dataset preview:")
display(training_df.head())
assert "total_discount_amount" not in training_df.columns
assert "maximum_discount_percentage" not in training_df.columns
assert "average_discount_percentage" not in training_df.columns
assert "order_discount_rate" not in training_df.columns

Number of model feature columns: 50

First 20 feature columns:
['order_month', 'order_day_of_week', 'order_hour', 'order_quarter', 'sales_channel', 'payment_method', 'currency', 'shipping_method', 'warehouse', 'marketing_channel', 'campaign_name', 'coupon_code', 'prior_order_count', 'prior_total_spend', 'prior_total_discount', 'prior_return_count', 'prior_average_order_value', 'prior_return_rate', 'prior_average_discount', 'days_since_previous_order']

Final dataset preview:


,order_id,order_timestamp,return_label,order_month,order_day_of_week,order_hour,order_quarter,sales_channel,payment_method,currency,shipping_method,warehouse,marketing_channel,campaign_name,coupon_code,customer_id,order_timestamp_customer,prior_order_count,prior_total_spend,prior_total_discount,prior_return_count,prior_average_order_value,prior_return_rate,prior_average_discount,days_since_previous_order,is_first_order,customer_age,gender,customer_segment,customer_state,customer_country,region,customer_acquisition_cost,item_count,unique_product_count,total_quantity,category_count,subcategory_count,brand_count,supplier_count,total_gross_sales,total_tax_amount,total_shipping_cost,total_item_net_sales,total_item_product_cost,total_item_profit,minimum_item_unit_price,maximum_item_unit_price,average_item_unit_price,minimum_product_rating,maximum_product_rating,average_product_rating,average_quantity_per_item,order_profit_margin_percentage,dominant_product_category
0,ORD-301242,2023-11-06 16:37:47,0,11,0,16,4,Mobile App,Digital Wallet,USD,Standard,WH-003,Direct,Default_Campaign,Unknown,CUST-003102,2023-11-06 16:37:47,5,"6,532.01",888.71,0,"1,306.40",0.00,177.74,312.12,0,55,Male,Consumer,Texas,USA,South,73.06,2,2,5,2,2,2,2,"1,350.19",61.07,12.03,945.40,588.33,345.04,244.40,287.13,265.76,3.00,4.80,3.90,2.50,36.50,Fashion
1,ORD-773460,2025-12-24 01:22:36,0,12,2,1,4,Website,Debit Card,EUR,Economy,WH-005,Direct,Default_Campaign,Unknown,CUST-003124,2025-12-24 01:22:36,9,"9,538.76","2,064.63",1,"1,059.86",0.11,229.40,317.40,0,26,Male,Premium,Baden-Württemberg,Germany,South,67.27,2,2,8,2,2,2,2,"3,144.64",320.83,9.00,"2,018.41","2,031.88",-22.47,31.36,754.80,393.08,3.50,4.00,3.75,4.00,-1.11,Jewelry
2,ORD-449374,2021-07-05 14:24:21,0,7,0,14,3,Mobile App,Cash on Delivery,USD,Express,WH-015,YouTube,Unknown,Unknown,CUST-012496,2021-07-05 14:24:21,0,0.00,0.00,0,NaN,NaN,NaN,NaN,1,69,Female,Premium,New York,USA,East,52.02,2,2,5,2,2,2,2,522.81,29.79,13.05,468.35,257.73,197.57,6.31,251.94,129.12,2.80,3.00,2.90,2.50,42.18,Health & Wellness
3,ORD-567636,2023-01-21 07:20:26,0,1,5,7,1,Social Media,Debit Card,USD,Express,WH-010,Email Marketing,Unknown,Unknown,CUST-023928,2023-01-21 07:20:26,4,"2,931.97",525.44,1,732.99,0.25,131.36,23.78,0,65,Male,Consumer,North Carolina,USA,South,65.91,2,2,2,2,2,2,2,544.62,34.39,20.85,546.52,277.84,247.83,264.43,280.19,272.31,3.80,4.60,4.20,1.00,45.35,Automotive
4,ORD-820028,2022-05-13 09:46:21,0,5,4,9,2,Social Media,Digital Wallet,INR,Standard,WH-013,Direct,Unknown,Unknown,CUST-012730,2022-05-13 09:46:21,0,0.00,0.00,0,NaN,NaN,NaN,NaN,1,34,Female,Consumer,Gujarat,India,West,66.26,3,3,6,3,3,3,3,"2,474.52",421.35,23.03,"2,785.27","1,231.88","1,530.36",62.26,587.18,237.66,2.90,4.20,3.70,2.00,54.94,Home Appliances


# Save Final Training Dataset

In [11]:
output_path = PROCESSED_DIR / "return_risk_training_data.csv"

training_df.to_csv(output_path, index=False)

print(f"Saved training dataset to: {output_path}")
print(f"Final shape: {training_df.shape}")

Saved training dataset to: C:\Users\Victus\OneDrive\Desktop\E-Commerce-Sales-Analytics\data\processed\return_risk_training_data.csv
Final shape: (123021, 55)


In [12]:
training_df.head()

,order_id,order_timestamp,return_label,order_month,order_day_of_week,order_hour,order_quarter,sales_channel,payment_method,currency,shipping_method,warehouse,marketing_channel,campaign_name,coupon_code,customer_id,order_timestamp_customer,prior_order_count,prior_total_spend,prior_total_discount,prior_return_count,prior_average_order_value,prior_return_rate,prior_average_discount,days_since_previous_order,is_first_order,customer_age,gender,customer_segment,customer_state,customer_country,region,customer_acquisition_cost,item_count,unique_product_count,total_quantity,category_count,subcategory_count,brand_count,supplier_count,total_gross_sales,total_tax_amount,total_shipping_cost,total_item_net_sales,total_item_product_cost,total_item_profit,minimum_item_unit_price,maximum_item_unit_price,average_item_unit_price,minimum_product_rating,maximum_product_rating,average_product_rating,average_quantity_per_item,order_profit_margin_percentage,dominant_product_category
0,ORD-301242,2023-11-06 16:37:47,0,11,0,16,4,Mobile App,Digital Wallet,USD,Standard,WH-003,Direct,Default_Campaign,Unknown,CUST-003102,2023-11-06 16:37:47,5,"6,532.01",888.71,0,"1,306.40",0.00,177.74,312.12,0,55,Male,Consumer,Texas,USA,South,73.06,2,2,5,2,2,2,2,"1,350.19",61.07,12.03,945.40,588.33,345.04,244.40,287.13,265.76,3.00,4.80,3.90,2.50,36.50,Fashion
1,ORD-773460,2025-12-24 01:22:36,0,12,2,1,4,Website,Debit Card,EUR,Economy,WH-005,Direct,Default_Campaign,Unknown,CUST-003124,2025-12-24 01:22:36,9,"9,538.76","2,064.63",1,"1,059.86",0.11,229.40,317.40,0,26,Male,Premium,Baden-Württemberg,Germany,South,67.27,2,2,8,2,2,2,2,"3,144.64",320.83,9.00,"2,018.41","2,031.88",-22.47,31.36,754.80,393.08,3.50,4.00,3.75,4.00,-1.11,Jewelry
2,ORD-449374,2021-07-05 14:24:21,0,7,0,14,3,Mobile App,Cash on Delivery,USD,Express,WH-015,YouTube,Unknown,Unknown,CUST-012496,2021-07-05 14:24:21,0,0.00,0.00,0,NaN,NaN,NaN,NaN,1,69,Female,Premium,New York,USA,East,52.02,2,2,5,2,2,2,2,522.81,29.79,13.05,468.35,257.73,197.57,6.31,251.94,129.12,2.80,3.00,2.90,2.50,42.18,Health & Wellness
3,ORD-567636,2023-01-21 07:20:26,0,1,5,7,1,Social Media,Debit Card,USD,Express,WH-010,Email Marketing,Unknown,Unknown,CUST-023928,2023-01-21 07:20:26,4,"2,931.97",525.44,1,732.99,0.25,131.36,23.78,0,65,Male,Consumer,North Carolina,USA,South,65.91,2,2,2,2,2,2,2,544.62,34.39,20.85,546.52,277.84,247.83,264.43,280.19,272.31,3.80,4.60,4.20,1.00,45.35,Automotive
4,ORD-820028,2022-05-13 09:46:21,0,5,4,9,2,Social Media,Digital Wallet,INR,Standard,WH-013,Direct,Unknown,Unknown,CUST-012730,2022-05-13 09:46:21,0,0.00,0.00,0,NaN,NaN,NaN,NaN,1,34,Female,Consumer,Gujarat,India,West,66.26,3,3,6,3,3,3,3,"2,474.52",421.35,23.03,"2,785.27","1,231.88","1,530.36",62.26,587.18,237.66,2.90,4.20,3.70,2.00,54.94,Home Appliances
